### 업로드 코드

In [ ]:
import pandas as pd
from pinecone import Pinecone
import base64
import numpy as np

# index table과의 연결
pc = Pinecone(api_key="api_key")
index = pc.Index(host="host")

In [2]:
# blob > vector
def parse_blob(encoded:str) -> np.ndarray:
    blob = base64.b64decode(encoded)
    return np.frombuffer(blob, dtype=np.float32)

# csv를 읽어서 pinecone 업로드할 vector로 변경
def tuple_to_vector(tup:tuple) -> dict:
    # 0: 제품코드, 1: 페이지, 2: index, 3: 설명, 4: 벡터
    buff = {}

    # 각 튜플 데이터를 읽어서 벡터 구조로 변경하는 코드
    metadata = {}
    metadata["product_code_header"] = tup[0][:3]
    metadata["product_code"] = tup[0]
    metadata["page_number"] = tup[1]
    metadata["index"] = tup[2]
    metadata["content"] = tup[3]
    
    buff["id"] = f"{tup[0]}_{tup[1]}_{tup[2]}"
    buff["values"] = parse_blob(tup[4]).tolist()
    buff["metadata"] = metadata

    return buff

"""
metadata_sample = {
    "product_code_header": "VAC", # 제품 분류 확인용 제품 코드
    "product_code": "VAC000", # 제품 코드
    "page_number": 3, # 페이지 번호
    "index": 1, # 같은 페이지에서 몇번째인지
    "content": "사용 설명서 내용은 여기에 길게 들어가서 나중에 검색할 때 어쩌구"
}
"""

"""
vector_sample =  {
    "id": "VAC000_3_1", # 코드_페이지번호_index
    "values": [], # parse_blob 활용해서 저장
    "metadata": metadata_sample
}
"""

'\nvector_sample =  {\n    "id": "VAC000_3_1", # 코드_페이지번호_index\n    "values": [], # parse_blob 활용해서 저장\n    "metadata": metadata_sample\n}\n'

In [4]:
# 로컬 dataframe 로드
dat = pd.read_csv("index/TV_index.csv").itertuples(index=False, name=None)

In [5]:
# 첫 번째 row만 테스트
first_row = next(dat)

# vector 변환
result = tuple_to_vector(first_row)

# 출력 확인
print("ID:")
print(result["id"])

print("\nMetadata:")
print(result["metadata"])

print("\nVector 앞 5개:")
print(result["values"][:5])

print("\nVector dimension:")
print(len(result["values"]))

ID:
TVT000_1_0

Metadata:
{'product_code_header': 'TVT', 'product_code': 'TVT000', 'page_number': 1, 'index': 0, 'content': '고객을 위해 우수한 품질의 서비스를 제공하는  기업에게 사단법인 한국서비스진흥협회에서  서비스 품질을 인증하는 마크로 LG전자는  서비스 품질 우수기업입니다. www.lge.com'}

Vector 앞 5개:
[0.0214996337890625, 0.0108184814453125, -0.05999755859375, -0.044189453125, 0.0164794921875]

Vector dimension:
1536


In [47]:
# vectors list 생성
vectors = []

for idx, d in enumerate(dat):

    try:
        vectors.append(tuple_to_vector(d))

    except Exception as e:
        print(f"{idx} 번째 row 에러")
        print(d[:4])
        print(e)
        break

In [48]:
# pinecone에 벡터 리스트 업로드

batch_size = 100

for i in range(0, len(vectors), batch_size):

    batch = vectors[i:i + batch_size]

    index.upsert(
        vectors=batch,
        namespace="user_manual"
    )

    print(f"{i} ~ {i + len(batch) - 1} 업로드 완료")

0 ~ 99 업로드 완료
100 ~ 199 업로드 완료
200 ~ 299 업로드 완료
300 ~ 399 업로드 완료
400 ~ 499 업로드 완료
500 ~ 599 업로드 완료
600 ~ 699 업로드 완료
700 ~ 799 업로드 완료
800 ~ 899 업로드 완료
900 ~ 999 업로드 완료
1000 ~ 1099 업로드 완료
1100 ~ 1199 업로드 완료
1200 ~ 1299 업로드 완료
1300 ~ 1399 업로드 완료
1400 ~ 1499 업로드 완료
1500 ~ 1599 업로드 완료
1600 ~ 1699 업로드 완료
1700 ~ 1799 업로드 완료
1800 ~ 1899 업로드 완료
1900 ~ 1999 업로드 완료
2000 ~ 2099 업로드 완료
2100 ~ 2199 업로드 완료
2200 ~ 2299 업로드 완료
2300 ~ 2399 업로드 완료
2400 ~ 2499 업로드 완료
2500 ~ 2599 업로드 완료
2600 ~ 2699 업로드 완료
2700 ~ 2799 업로드 완료
2800 ~ 2899 업로드 완료
2900 ~ 2999 업로드 완료
3000 ~ 3099 업로드 완료
3100 ~ 3199 업로드 완료
3200 ~ 3299 업로드 완료
3300 ~ 3399 업로드 완료
3400 ~ 3499 업로드 완료
3500 ~ 3599 업로드 완료
3600 ~ 3699 업로드 완료
3700 ~ 3799 업로드 완료
3800 ~ 3899 업로드 완료
3900 ~ 3999 업로드 완료
4000 ~ 4099 업로드 완료
4100 ~ 4199 업로드 완료
4200 ~ 4299 업로드 완료
4300 ~ 4399 업로드 완료
4400 ~ 4499 업로드 완료
4500 ~ 4599 업로드 완료
4600 ~ 4699 업로드 완료
4700 ~ 4799 업로드 완료
4800 ~ 4899 업로드 완료
4900 ~ 4999 업로드 완료
5000 ~ 5099 업로드 완료
5100 ~ 5199 업로드 완료
5200 ~ 5299 업로드 완료
5300 ~ 5399 업로드 

### 검색 예제

In [15]:
df = pd.read_csv("index/AC_index.csv")

encoded = df.iloc[3954]["embedding_blob"]

sample_vector = parse_blob(encoded)

print(sample_vector[:5])
print(len(sample_vector))

[ 0.00080776 -0.01444244 -0.05557251  0.00858307  0.00072575]
1536


In [43]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

In [62]:
query = "풍향 조절"

response = client.embeddings.create(
    model="text-embedding-3-small",
    input=query
)

query_vector = response.data[0].embedding

In [63]:
# 실제 검색
result = index.query(
    namespace="user_manual",

    vector=query_vector,

    top_k=10,

    include_metadata=True
)


# 검색 결과 출력
for i, match in enumerate(result["matches"]):

    print(f"\n===== 검색 결과 {i+1} =====")

    print("ID:")
    print(match["id"])

    print("\nScore:")
    print(match["score"])

    print("\n제품 코드:")
    print(match["metadata"]["product_code"])

    print("\n페이지:")
    print(match["metadata"]["page_number"])

    print("\nIndex:")
    print(match["metadata"]["index"])

    print("\nContent:")
    print(match["metadata"]["content"])


===== 검색 결과 1 =====
ID:
ACT230_33_5

Score:
0.692406714

제품 코드:
ACT230

페이지:
33

Index:
5

Content:
풍향 조절기가 위로  올라갑니다. 

===== 검색 결과 2 =====
ID:
ACT229_33_5

Score:
0.692406714

제품 코드:
ACT229

페이지:
33

Index:
5

Content:
풍향 조절기가 위로  올라갑니다. 

===== 검색 결과 3 =====
ID:
ACT224_33_5

Score:
0.692406714

제품 코드:
ACT224

페이지:
33

Index:
5

Content:
풍향 조절기가 위로  올라갑니다. 

===== 검색 결과 4 =====
ID:
ACT228_33_5

Score:
0.691962123

제품 코드:
ACT228

페이지:
33

Index:
5

Content:
풍향 조절기가 위로  올라갑니다. 

===== 검색 결과 5 =====
ID:
ACT185_13_3

Score:
0.658211052

제품 코드:
ACT185

페이지:
13

Index:
3

Content:
우 풍향 조절기 • 바람 방향을 왼쪽, 오른쪽으로 조절할 수 있습니다. 

===== 검색 결과 6 =====
ID:
ACT226_54_6

Score:
0.591322899

제품 코드:
ACT226

페이지:
54

Index:
6

Content:
하  풍향 조절기가 열리고 청소하기 쉽게 고정됩니다. 

===== 검색 결과 7 =====
ID:
ACT226_37_5

Score:
0.570157111

제품 코드:
ACT226

페이지:
37

Index:
5

Content:
나며 상하 풍향 조절기가  닫힙니다. 

===== 검색 결과 8 =====
ID:
ACT193_13_2

Score:
0.566758692

제품 코드:
ACT193

페이지:
13

Index:
2

Content:
기 • 바람 방향을 왼쪽, 오른쪽